# Variational Quantum Classifier (VQC): A First-Principles 2-Qubit Controlled Experiment

**Author / Researcher:** Quantum Machine Learning Implementation & Investigation  
**Environment:** Python 3 (NumPy, SciPy, Matplotlib, Scikit-Learn) — *No quantum frameworks used*  
**Scope:** Sections 01 to 10 (Objectives through Quantum Measurement Layer)

---


# 01 — Research Objective & Experimental Design

## 1.1 What is a Variational Quantum Classifier (VQC)?
A **Variational Quantum Classifier (VQC)** is a hybrid quantum-classical machine learning algorithm designed for supervised classification tasks. The overall computational workflow consists of:
1. **Classical-to-Quantum Feature Encoding ($U_{\text{enc}}(x)$):** Continuous classical input vectors $x \in \mathbb{R}^d$ are mapped into quantum states $|\psi(x)\rangle$ in a $2^n$-dimensional Hilbert space $\mathcal{H} = (\mathbb{C}^2)^{\otimes n}$.
2. **Parameterized Variational Circuit ($U(\theta)$):** A sequence of parameterized rotation gates and entangling operations applies a parameterized unitary transformation $|\psi(x, \theta)\rangle = U(\theta) |\psi(x)\rangle$.
3. **Quantum Measurement (Observable Expectation $\langle O \rangle$):** One or more Hermitian observables $O$ are measured with respect to the state $|\psi(x, \theta)\rangle$, yielding expectation values $\langle O \rangle \in \mathbb{R}$.
4. **Classical Post-Processing & Loss Evaluation:** Expectation values are mapped to predicted class probabilities $\hat{y} \in [0, 1]$ and evaluated against ground-truth labels $y$ using a classical loss function $\mathcal{L}(\hat{y}, y)$.
5. **Classical Optimization:** Gradients of the loss with respect to circuit parameters $\theta$ are calculated (analytically via the parameter-shift rule or numerically via finite differences) and updated on classical hardware.

```
Classical Input x ──► [Feature Map U_enc(x)] ──► [Variational Ansatz U(θ)] ──► [Measurement <O>] ──► Prediction y_hat
                               ▲                                                                           │
                               │                                                                           ▼
                       Classical Optimizer ◄────────────────── Compute Gradients ◄────────────── Loss L(y_hat, y)
```

---

## 1.2 Purpose of This Experiment
The primary purpose of this research notebook is to **construct, verify, and empirically study a complete VQC pipeline from first mathematical principles using only standard Python and NumPy**. 

By deliberately eliminating high-level quantum abstraction frameworks (such as PennyLane, Qiskit, or Cirq), we expose and verify every mathematical step:
- Exact complex state-vector linear algebra;
- Tensor-product construction of multi-qubit operators;
- Rotation gate generator algebra and parameter-shift gradients;
- Empirical trade-offs between depth, trainability, noise, and error mitigation.

> **Crucial Disclaimer Regarding Quantum Advantage:**  
> This experiment is designed exclusively for **scientific transparency, mechanistic understanding, and controlled empirical analysis**. A 2-qubit quantum system simulated classically on standard hardware does not and cannot provide quantum advantage over classical algorithms. We make no claims of quantum supremacy or superiority over classical models.

---

## 1.3 Why Study an Intentionally Constrained 2-Qubit System?
This experiment deliberately restricts the quantum system to **$N=2$ qubits** and **$D=2$ classical features**. This design decision is grounded in specific scientific and methodological justifications:
1. **Minimal Non-Trivial Hilbert Space for Entanglement:** A 1-qubit system ($2\times 2$ matrices) cannot exhibit entanglement. A 2-qubit system ($4\times 4$ matrices) is the strictly minimal system capable of demonstrating non-local quantum correlations via the Controlled-NOT (CNOT) gate.
2. **Direct Visualizability in 2D Feature Space:** With exactly two input features $(x_1, x_2)$, the input space, class distributions, and resulting quantum decision boundaries can be plotted and visually analyzed without lossy dimensionality reduction (such as PCA or t-SNE).
3. **One-to-One Feature-to-Qubit Mapping:** Mapping $x_1 \to q_0$ and $x_2 \to q_1$ keeps the encoding completely transparent and avoids confounding encoding artifacts with variational ansatz expressibility.
4. **Uncompromised Mathematical Auditability:** In a 4-dimensional state space, every state vector, density matrix, unitary matrix, and gradient can be explicitly written down, inspected, and validated against analytical ground truths without computational bottlenecks or numerical approximations.

---

## 1.4 Research Questions
Throughout this structured experiment, we formulate and address five specific research questions:
- **RQ1 (Expressibility & Decision Boundaries):** Can a minimal 2-qubit variational circuit with $R_Y$ angle encoding and a single entangling layer learn a non-linear decision boundary for a 2D binary classification task?
- **RQ2 (Gradient Exactness):** Does the analytical parameter-shift rule produce exact gradients matching central finite differences under state-vector simulation, and what are the numerical limits of this agreement?
- **RQ3 (Optimizer Dynamics):** How does standard Gradient Descent compare with an adaptive moment estimator (Adam) in terms of loss convergence rate, parameter trajectories, and gradient norm stability in a quantum loss landscape?
- **RQ4 (Circuit Depth & Trainability):** As variational depth increases from $L=1$ to $L=2$ and $L=3$, how do expressive capacity, parameter count, and gradient norms evolve, and do we observe early precursors to trainability degradation?
- **RQ5 (Finite Sampling, Noise, and Mitigation):** How do finite measurement shot noise ($S \in \{100, 1000, 10000\}$) and quantum noise channels (depolarizing and readout) corrupt expectation values and decision boundaries, and to what extent can Zero-Noise Extrapolation (ZNE) and readout error mitigation recover fidelity?

---

## 1.5 Experimental Constraints & Controlled Design

### Baseline Configuration
The experiment establishes a strict, controlled baseline configuration against which all subsequent experimental variations are compared:
- **Qubits ($N_{\text{qubits}}$):** 2
- **Features ($N_{\text{features}}$):** 2 continuous classical inputs
- **Encoding Scheme:** Product $R_Y$ angle encoding: $U_{\text{enc}}(x) = R_Y(x_1) \otimes R_Y(x_2)$
- **Ansatz Architecture:** 1 variational layer comprising single-qubit rotations $R_Y(\theta) R_Z(\theta)$ on each qubit followed by an entangling $CNOT_{0 \to 1}$ gate (4 parameters total)
- **Measurement Observable:** Pauli-$Z$ on qubit 0: $O = Z_0 = Z \otimes I$
- **Probability Mapping:** Linear expectation-to-probability mapping $\hat{y} = (\langle O \rangle + 1) / 2$
- **Loss Function:** Binary Cross-Entropy (BCE) with numerical clipping safeguards ($\epsilon = 10^{-15}$)
- **Gradient Calculation:** Analytical parameter-shift rule (shift $s = \pi/2$)
- **Baseline Optimizer:** Standard Gradient Descent (learning rate $\eta = 0.1$)
- **Simulation Engine:** Exact state-vector complex linear algebra (noiseless, infinite-shot)

### Controlled Variables Across All Experiments
To ensure scientific validity, the following variables are strictly held constant unless explicitly designated as the independent variable of a specific study:
1. **Random Seed:** `SEED = 42` across dataset generation, train/val/test splits, parameter initialization, and stochastic sampling.
2. **Dataset Partitioning:** Fixed train (70%), validation (15%), and test (15%) splits. Test data is held out strictly for final evaluation.
3. **Evaluation Metrics:** Evaluated consistently using BCE Loss, Classification Accuracy, Precision, Recall, and Macro F1-score.

### Experimental Variables Studied in Later Sections
- **Optimizer Comparison:** Gradient Descent vs. Adam (Sections 14, 17, 18).
- **Ansatz Depth Study:** $L \in \{1, 2, 3\}$ layers (Section 19).
- **Measurement Statistics:** Exact expectation values vs. Finite-shot sampling with $S \in \{100, 1000, 10000\}$ (Section 20).
- **Quantum Noise:** Noiseless vs. Depolarizing channel vs. Readout error (Section 21).
- **Error Mitigation:** Unmitigated vs. Readout calibration matrix inversion and Zero-Noise Extrapolation (Sections 22, 23).


# 02 — Imports & Configuration

In this section, we import all required classical dependencies and establish centralized configuration constants. 

### Methodological Principles:
1. **Explicit Classical Dependencies:** We import only foundational libraries: `numpy` for linear algebra, `pandas` for tabular data handling, `matplotlib` for scientific visualization, and `scikit-learn` strictly for dataset splitting and classification metrics.
2. **Zero Quantum Frameworks:** No pre-built quantum simulation libraries (`qiskit`, `pennylane`, `cirq`, etc.) are imported.
3. **Centralized Configuration:** All hyperparameters, architectural dimensions, and random seeds are declared once at the top of the notebook to eliminate magic numbers and ensure strict experimental reproducibility.
4. **Environment Verification:** We programmatically verify that execution occurs within the designated `.venv` virtual environment and log all package versions.


In [1]:
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ---------------------------------------------------------------------------
# Centralized Experimental Configuration
# ---------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)

# Problem and Circuit Architecture Dimensions
N_QUBITS = 2        # Number of quantum bits (Hilbert space dimension = 2^2 = 4)
N_FEATURES = 2      # Number of continuous classical input features
N_LAYERS = 1        # Baseline variational ansatz depth (1 layer: 4 parameters)

# Optimization Hyperparameters
LEARNING_RATE = 0.1 # Baseline gradient descent learning rate (eta)
N_EPOCHS = 30       # Baseline number of training iterations


### Verification: Environment and Configuration

**What should be true:**
1. The Python executable path must confirm execution from within the project's local virtual environment (`.venv`).
2. The core classical dependencies (`numpy`, `pandas`, `matplotlib`, `scikit-learn`) must be successfully imported with recorded versions.
3. The centralized configuration variables (`SEED`, `N_QUBITS`, `N_FEATURES`, `N_LAYERS`, `LEARNING_RATE`, `N_EPOCHS`) must match the experimental design specification exactly.


In [2]:
# ---------------------------------------------------------------------------
# Environment & Configuration Verification
# ---------------------------------------------------------------------------
print("=" * 70)
print("SYSTEM & RUNTIME ENVIRONMENT")
print("=" * 70)
print(f"Python Executable : {sys.executable}")
print(f"Python Version    : {platform.python_version()}")
print(f"Operating System  : {platform.system()} {platform.release()} ({platform.architecture()[0]})")
print("-" * 70)
print("INSTALLED DEPENDENCY VERSIONS")
print("-" * 70)
print(f"NumPy Version     : {np.__version__}")
print(f"Pandas Version    : {pd.__version__}")
print(f"Matplotlib Version: {plt.matplotlib.__version__}")
import sklearn
print(f"Scikit-Learn      : {sklearn.__version__}")
print("-" * 70)
print("EXPERIMENTAL CONFIGURATION PARAMETERS")
print("-" * 70)
print(f"  Random Seed (SEED)           : {SEED}")
print(f"  Qubit Count (N_QUBITS)       : {N_QUBITS}")
print(f"  Feature Count (N_FEATURES)   : {N_FEATURES}")
print(f"  Ansatz Depth (N_LAYERS)      : {N_LAYERS}")
print(f"  Learning Rate (LEARNING_RATE): {LEARNING_RATE}")
print(f"  Epoch Count (N_EPOCHS)       : {N_EPOCHS}")
print("=" * 70)

# Verification Assertions
assert ".venv" in sys.executable.lower(), (
    f"Environment Assertion Failed: Expected execution inside '.venv', but got: {sys.executable}"
)
assert N_QUBITS == 2, f"Configuration Assertion Failed: Expected N_QUBITS=2, got {N_QUBITS}"
assert N_FEATURES == 2, f"Configuration Assertion Failed: Expected N_FEATURES=2, got {N_FEATURES}"
assert N_LAYERS == 1, f"Configuration Assertion Failed: Expected baseline N_LAYERS=1, got {N_LAYERS}"
assert LEARNING_RATE == 0.1, f"Configuration Assertion Failed: Expected LEARNING_RATE=0.1, got {LEARNING_RATE}"
assert N_EPOCHS == 30, f"Configuration Assertion Failed: Expected N_EPOCHS=30, got {N_EPOCHS}"
assert SEED == 42, f"Configuration Assertion Failed: Expected SEED=42, got {SEED}"

print("\n[VERIFICATION PASSED] Environment validated inside .venv and configuration initialized.")


SYSTEM & RUNTIME ENVIRONMENT
Python Executable : C:\Users\yessi\OneDrive\Documentos\VQC_Implementation\.venv\Scripts\python.exe
Python Version    : 3.13.5
Operating System  : Windows 11 (64bit)
----------------------------------------------------------------------
INSTALLED DEPENDENCY VERSIONS
----------------------------------------------------------------------
NumPy Version     : 2.5.3
Pandas Version    : 3.0.5
Matplotlib Version: 3.11.2
Scikit-Learn      : 1.9.1
----------------------------------------------------------------------
EXPERIMENTAL CONFIGURATION PARAMETERS
----------------------------------------------------------------------
  Random Seed (SEED)           : 42
  Qubit Count (N_QUBITS)       : 2
  Feature Count (N_FEATURES)   : 2
  Ansatz Depth (N_LAYERS)      : 1
  Learning Rate (LEARNING_RATE): 0.1
  Epoch Count (N_EPOCHS)       : 30

[VERIFICATION PASSED] Environment validated inside .venv and configuration initialized.


### Interpretation of Environment Verification
The runtime environment verification confirms that the notebook is executing within the dedicated `.venv` virtual environment. The classical computational stack consists of modern, stable releases (`numpy 2.x`, `pandas 3.x`, `matplotlib 3.x`, `scikit-learn 1.x`). All experimental hyperparameters are locked to the baseline protocol with a fixed pseudo-random seed (`SEED = 42`).


# 03 — Mathematical Foundations

Before implementing quantum states and gates, we establish the rigorous mathematical framework governing every operation in this notebook. The mathematics formulated here will translate directly into NumPy linear algebra.

---

## 3.1 Hilbert Spaces & Computational Basis States
In quantum mechanics, the state of an $n$-qubit quantum system is represented by a unit vector residing in a $2^n$-dimensional complex Hilbert space:
$$\mathcal{H} = (\mathbb{C}^2)^{\otimes n}$$

For a single qubit ($n=1$), $\mathcal{H}_1 \cong \mathbb{C}^2$, spanned by the orthonormal computational basis vectors:
$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

For our 2-qubit system ($n=2$), $\mathcal{H} = \mathcal{H}_1 \otimes \mathcal{H}_2 \cong \mathbb{C}^4$. The standard computational basis is formed by the tensor product of the single-qubit basis states under the **standard Kronecker product ordering**:
$$|q_0 q_1\rangle \equiv |q_0\rangle \otimes |q_1\rangle$$

Specifically:
$$|00\rangle = |0\rangle \otimes |0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \otimes \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix}$$
$$|01\rangle = |0\rangle \otimes |1\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \otimes \begin{pmatrix} 0 \\ 1 \end{pmatrix} = \begin{pmatrix} 0 \\ 1 \\ 0 \\ 0 \end{pmatrix}$$
$$|10\rangle = |1\rangle \otimes |0\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix} \otimes \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix}$$
$$|11\rangle = |1\rangle \otimes |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix} \otimes \begin{pmatrix} 0 \\ 1 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$

> **Explicit Qubit Ordering Convention:**  
> We define **Qubit 0 ($q_0$) as the first (leftmost / most significant) tensor factor** and **Qubit 1 ($q_1$) as the second (rightmost / least significant) tensor factor**.  
> The computational basis state $|q_0 q_1\rangle$ corresponds to integer index $i = 2 q_0 + q_1 \in \{0, 1, 2, 3\}$. This convention is maintained consistently throughout all multi-qubit gates, observables, and state vectors.

---

## 3.2 Tensor (Kronecker) Products & Multi-Qubit Operators
Given linear operators $A \in \mathbb{C}^{m \times n}$ and $B \in \mathbb{C}^{p \times q}$, their tensor product $A \otimes B \in \mathbb{C}^{mp \times nq}$ is defined via the Kronecker product:
$$A \otimes B = \begin{pmatrix}
A_{11} B & A_{12} B & \cdots & A_{1n} B \\
A_{21} B & A_{22} B & \cdots & A_{2n} B \\
\vdots & \vdots & \ddots & \vdots \\
A_{m1} B & A_{m2} B & \cdots & A_{mn} B
\end{pmatrix}$$

### Essential Algebraic Properties:
1. **Bilinearity:** $(\alpha A) \otimes B = A \otimes (\alpha B) = \alpha (A \otimes B)$ for any $\alpha \in \mathbb{C}$.
2. **Mixed-Product Property:** If products $AC$ and $BD$ are defined, then:
   $$(A \otimes B)(C \otimes D) = (AC) \otimes (BD)$$
3. **Conjugate Transposition:** $(A \otimes B)^\dagger = A^\dagger \otimes B^\dagger$.
4. **Unitarity Preservation:** If $A$ and $B$ are unitary, then $A \otimes B$ is unitary:
   $$(A \otimes B)^\dagger (A \otimes B) = (A^\dagger A) \otimes (B^\dagger B) = I_m \otimes I_p = I_{mp}$$

---

## 3.3 Complex State Vectors & Normalization
An arbitrary pure state $|\psi\rangle \in \mathbb{C}^4$ of a 2-qubit system is expressed as a complex superposition over the computational basis:
$$|\psi\rangle = c_{00} |00\rangle + c_{01} |01\rangle + c_{10} |10\rangle + c_{11} |11\rangle = \begin{pmatrix} c_{00} \\ c_{01} \\ c_{10} \\ c_{11} \end{pmatrix}, \quad c_{jk} \in \mathbb{C}$$

By Born's rule, the probability of measuring the system in basis state $|jk\rangle$ is given by the squared magnitude of its probability amplitude:
$$P(jk) = |\langle jk | \psi \rangle|^2 = |c_{jk}|^2$$

Conservation of total probability requires that $|\psi\rangle$ satisfies the normalization condition:
$$\langle \psi | \psi \rangle = \sum_{j=0}^1 \sum_{k=0}^1 |c_{jk}|^2 = |c_{00}|^2 + |c_{01}|^2 + |c_{10}|^2 + |c_{11}|^2 = 1$$

---

## 3.4 Unitary Transformations & Quantum Evolution
Under the postulates of quantum mechanics, any closed quantum system evolves deterministically via linear, norm-preserving operators termed **unitary matrices** $U$:
$$U^\dagger U = U U^\dagger = I$$
where $U^\dagger = (U^*)^T$ is the conjugate transpose (Hermitian adjoint) of $U$.

When an initial state $|\psi\rangle$ undergoes unitary evolution via $U$, the transformed state is:
$$|\psi'\rangle = U |\psi\rangle$$

Because $U$ is unitary, the inner product and norm are strictly invariant:
$$\langle \psi' | \psi' \rangle = (U |\psi\rangle)^\dagger (U |\psi\rangle) = \langle \psi | U^\dagger U | \psi \rangle = \langle \psi | I | \psi \rangle = \langle \psi | \psi \rangle = 1$$

---

## 3.5 Parameterized Rotation Gates & Pauli Generators
Quantum circuits are parameterized using continuous single-qubit rotation gates generated by the Hermitian Pauli matrices:
$$I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad
X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad
Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad
Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

### Properties of Pauli Matrices:
- **Hermiticity:** $P^\dagger = P$ for $P \in \{I, X, Y, Z\}$.
- **Involutory (Square to Identity):** $X^2 = Y^2 = Z^2 = I$.
- **Trace-Zero:** $\text{Tr}(X) = \text{Tr}(Y) = \text{Tr}(Z) = 0$.
- **Commutation and Anti-Commutation:**
  $$[X, Y] = XY - YX = 2iZ, \quad \{X, Y\} = XY + YX = 0$$

### Derivation of Parameterized Rotations via Matrix Exponentials:
A rotation about axis $P \in \{X, Y, Z\}$ by angle $\theta$ is defined via the matrix exponential:
$$R_P(\theta) \equiv \exp\left(-i \frac{\theta}{2} P\right) = \sum_{k=0}^\infty \frac{1}{k!} \left(-i \frac{\theta}{2} P\right)^k$$

Separating the Taylor series into even ($k=2m$) and odd ($k=2m+1$) powers, and using $P^{2m} = I$ and $P^{2m+1} = P$:
$$R_P(\theta) = \left[ \sum_{m=0}^\infty \frac{(-1)^m}{(2m)!} \left(\frac{\theta}{2}\right)^{2m} \right] I - i \left[ \sum_{m=0}^\infty \frac{(-1)^m}{(2m+1)!} \left(\frac{\theta}{2}\right)^{2m+1} \right] P$$
$$R_P(\theta) = \cos\left(\frac{\theta}{2}\right) I - i \sin\left(\frac{\theta}{2}\right) P$$

### Explicit Closed Matrix Forms:
1. **$R_Y(\theta)$ Rotation Gate:**
   $$R_Y(\theta) = \cos\left(\frac{\theta}{2}\right) \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} - i \sin\left(\frac{\theta}{2}\right) \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} = \begin{pmatrix} \cos(\theta/2) & -\sin(\theta/2) \\ \sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$$
   *Notice that $R_Y(\theta)$ is purely real-valued.*

2. **$R_Z(\theta)$ Rotation Gate:**
   $$R_Z(\theta) = \cos\left(\frac{\theta}{2}\right) \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} - i \sin\left(\frac{\theta}{2}\right) \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} = \begin{pmatrix} \cos(\theta/2) - i\sin(\theta/2) & 0 \\ 0 & \cos(\theta/2) + i\sin(\theta/2) \end{pmatrix} = \begin{pmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2} \end{pmatrix}$$

Both gates are unitary: $R_Y(\theta)^\dagger R_Y(\theta) = I$ and $R_Z(\theta)^\dagger R_Z(\theta) = I$.

---

## 3.6 Quantum Observables & Expectation Values
A physical measurement in quantum mechanics is defined by a Hermitian observable operator $O = O^\dagger$. By the spectral theorem, $O$ admits an eigendecomposition $O = \sum_k \lambda_k |\phi_k\rangle \langle \phi_k|$ with real eigenvalues $\lambda_k \in \mathbb{R}$.

The **expectation value** of $O$ with respect to state $|\psi\rangle$ is:
$$\langle O \rangle = \langle \psi | O | \psi \rangle = \psi^\dagger O \psi$$

### Pauli-Z Measurement Convention:
In our baseline 2-qubit VQC, we measure the Pauli-$Z$ observable on qubit 0:
$$O = Z_0 \equiv Z \otimes I = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \otimes \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & -1 & 0 \\
0 & 0 & 0 & -1
\end{pmatrix}$$

Expanding in terms of state amplitudes $|\psi\rangle = (c_{00}, c_{01}, c_{10}, c_{11})^T$:
$$\langle Z_0 \rangle = \langle \psi | (Z \otimes I) | \psi \rangle = |c_{00}|^2 + |c_{01}|^2 - |c_{10}|^2 - |c_{11}|^2 \in [-1, +1]$$

### Mapping Expectation Values to Classification Probabilities:
Binary cross-entropy loss requires predicted probabilities $\hat{y} \in [0, 1]$. Because $\langle Z_0 \rangle \in [-1, +1]$, we apply the affine transformation:
$$\hat{y} = \frac{\langle Z_0 \rangle + 1}{2}$$

- When $\langle Z_0 \rangle = +1$ (qubit 0 is definitely in state $|0\rangle$), $\hat{y} = 1$.
- When $\langle Z_0 \rangle = -1$ (qubit 0 is definitely in state $|1\rangle$), $\hat{y} = 0$.
- When $\langle Z_0 \rangle = 0$ (maximal superposition/uncertainty), $\hat{y} = 0.5$.

---

## 3.7 The Complete VQC Pipeline
Combining the mathematical components, the end-to-end forward pass for a single classical input $x = (x_1, x_2)^T$ is:
1. **Initial Ground State:** $|\psi_0\rangle = |00\rangle$.
2. **Feature Map State:** $|\psi(x)\rangle = U_{\text{enc}}(x) |00\rangle = (R_Y(x_1) \otimes R_Y(x_2)) |00\rangle$.
3. **Parameterized Variational State:** $|\psi(x, \theta)\rangle = U(\theta) |\psi(x)\rangle$.
4. **Observable Expectation:** $\langle Z_0(x, \theta) \rangle = \langle \psi(x, \theta) | (Z \otimes I) | \psi(x, \theta) \rangle$.
5. **Class Probability Prediction:** $\hat{y} = \frac{\langle Z_0(x, \theta) \rangle + 1}{2}$.
6. **Binary Cross-Entropy Loss:**
   $$\mathcal{L}(\hat{y}, y) = - \left[ y \log(\hat{y} + \epsilon) + (1 - y) \log(1 - \hat{y} + \epsilon) \right]$$

---

## 3.8 Optimization & The Analytical Parameter-Shift Rule
To train the variational parameters $\theta$ via gradient-based optimization, we require partial derivatives $\frac{\partial \mathcal{L}}{\partial \theta_j}$.

By the chain rule:
$$\frac{\partial \mathcal{L}}{\partial \theta_j} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial \langle Z_0 \rangle} \cdot \frac{\partial \langle Z_0 \rangle}{\partial \theta_j} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{1}{2} \cdot \frac{\partial \langle Z_0 \rangle}{\partial \theta_j}$$

On physical quantum hardware, one cannot apply automatic differentiation (backpropagation) because intermediate quantum states cannot be cloned or measured without wave-function collapse (No-Cloning Theorem). 

Instead, we utilize the **Parameter-Shift Rule** (Mitarai et al., 2018; Schuld et al., 2019), which yields **exact analytical gradients on quantum circuits**.

### Derivation of the Parameter-Shift Rule:
Consider an expectation value $f(\theta) = \langle \psi | U^\dagger(\theta) O U(\theta) | \psi \rangle$ where parameter $\theta$ enters through a unitary gate $G(\theta) = \exp(-i \frac{\theta}{2} P)$ generated by a Pauli operator $P$ with eigenvalues $\pm 1$.

Because $P^2 = I$, the unitary gate can be written as:
$$G(\theta) = \cos\left(\frac{\theta}{2}\right) I - i \sin\left(\frac{\theta}{2}\right) P$$

Substituting this into $f(\theta)$, the expectation value can be expanded as a finite trigonometric polynomial in $\theta$:
$$f(\theta) = A + B \cos(\theta) + C \sin(\theta)$$
where $A, B, C \in \mathbb{R}$ are constant coefficients determined by the state and remaining gates.

Taking the direct analytical derivative with respect to $\theta$:
$$\frac{d f}{d \theta} = - B \sin(\theta) + C \cos(\theta)$$

Now evaluate $f(\theta)$ at shifted parameters $\theta + s$ and $\theta - s$, choosing shift $s = \frac{\pi}{2}$:
$$f\left(\theta + \frac{\pi}{2}\right) = A + B \cos\left(\theta + \frac{\pi}{2}\right) + C \sin\left(\theta + \frac{\pi}{2}\right) = A - B \sin(\theta) + C \cos(\theta)$$
$$f\left(\theta - \frac{\pi}{2}\right) = A + B \cos\left(\theta - \frac{\pi}{2}\right) + C \sin\left(\theta - \frac{\pi}{2}\right) = A + B \sin(\theta) - C \cos(\theta)$$

Subtracting the two shifted evaluations:
$$f\left(\theta + \frac{\pi}{2}\right) - f\left(\theta - \frac{\pi}{2}\right) = - 2 B \sin(\theta) + 2 C \cos(\theta) = 2 \frac{d f}{d \theta}$$

Dividing by 2 yields the **Parameter-Shift Rule**:
$$\frac{\partial \langle O \rangle}{\partial \theta} = \frac{1}{2} \left[ \langle O \rangle\left(\theta + \frac{\pi}{2}\right) - \langle O \rangle\left(\theta - \frac{\pi}{2}\right) \right]$$

### Key Advantages over Classical Finite Differences:
Unlike central finite differences $\frac{f(\theta + \epsilon) - f(\theta - \epsilon)}{2\epsilon}$, which:
- Incur $O(\epsilon^2)$ approximation truncation error,
- Suffer severe numerical cancellation and catastrophic loss of significance when $\epsilon \to 0$,
- Break down completely in the presence of shot noise or quantum hardware fluctuations,

the **parameter-shift rule is an exact analytical identity** using macroscopic shifts ($s = \pi/2$). It evaluates the exact gradient without truncation error!


### Verification: Mathematical Identities & Algebraic Grounding

**What should be true:**
To verify our mathematical foundations before constructing the notebook's circuit code in Section 04:
1. **Pauli Algebra:** Pauli matrices must satisfy $X^2 = Y^2 = Z^2 = I$, anti-commutation $\{X, Y\} = 0$, and commutation $[X, Y] = 2iZ$.
2. **Matrix Exponential Expansion:** The Taylor series matrix exponential $\exp(-i \frac{\theta}{2} Y)$ must identically match our closed-form matrix for $R_Y(\theta)$ across test angles.
3. **Unitarity:** $R_Y(\theta)$ and $R_Z(\theta)$ must satisfy $U^\dagger U = I$ within machine precision ($< 10^{-15}$).
4. **Mixed-Product Property:** Multi-qubit tensor products must satisfy $(A \otimes B)(C \otimes D) = (AC) \otimes (BD)$.
5. **Parameter-Shift Trigonometric Identity:** For any $f(\theta) = A + B\cos(\theta) + C\sin(\theta)$, the parameter-shift formula $\frac{1}{2}[f(\theta + \pi/2) - f(\theta - \pi/2)]$ must equal the exact analytical derivative $f'(\theta)$ to machine precision.


In [3]:
# ---------------------------------------------------------------------------
# Numerical Verification of Mathematical Foundations
# ---------------------------------------------------------------------------
# Define basic Pauli matrices for algebraic verification
I_mat = np.array([[1, 0], [0, 1]], dtype=complex)
X_mat = np.array([[0, 1], [1, 0]], dtype=complex)
Y_mat = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z_mat = np.array([[1, 0], [0, -1]], dtype=complex)

# 1. Pauli Involutory & Commutation Verification
assert np.allclose(X_mat @ X_mat, I_mat), "Failed: X^2 != I"
assert np.allclose(Y_mat @ Y_mat, I_mat), "Failed: Y^2 != I"
assert np.allclose(Z_mat @ Z_mat, I_mat), "Failed: Z^2 != I"
assert np.allclose(X_mat @ Y_mat + Y_mat @ X_mat, np.zeros((2, 2))), "Failed: {X, Y} != 0"
assert np.allclose(X_mat @ Y_mat - Y_mat @ X_mat, 2j * Z_mat), "Failed: [X, Y] != 2iZ"
print("[VERIFIED 1/5] Pauli matrix algebraic and commutation identities hold exactly.")

# 2. Matrix Exponential vs Closed-Form Rotation Gate
from scipy.linalg import expm

test_angles = [0.0, np.pi / 6, np.pi / 4, np.pi / 2, np.pi, 2.345]
for theta in test_angles:
    # Taylor matrix exponential exp(-i * theta/2 * Y)
    exp_Y = expm(-0.5j * theta * Y_mat)
    # Closed form: [[cos(t/2), -sin(t/2)], [sin(t/2), cos(t/2)]]
    closed_RY = np.array([
        [np.cos(theta / 2), -np.sin(theta / 2)],
        [np.sin(theta / 2),  np.cos(theta / 2)]
    ], dtype=complex)
    assert np.allclose(exp_Y, closed_RY, atol=1e-15), f"Failed RY closed-form at theta={theta}"

    # Taylor matrix exponential exp(-i * theta/2 * Z)
    exp_Z = expm(-0.5j * theta * Z_mat)
    # Closed form: diag(e^{-i t/2}, e^{i t/2})
    closed_RZ = np.array([
        [np.exp(-0.5j * theta), 0.0],
        [0.0, np.exp(0.5j * theta)]
    ], dtype=complex)
    assert np.allclose(exp_Z, closed_RZ, atol=1e-15), f"Failed RZ closed-form at theta={theta}"

print("[VERIFIED 2/5] Matrix exponentials exp(-i*theta/2*P) match closed-form RY and RZ gates.")

# 3. Unitarity of Rotation Gates
for theta in test_angles:
    closed_RY = np.array([
        [np.cos(theta / 2), -np.sin(theta / 2)],
        [np.sin(theta / 2),  np.cos(theta / 2)]
    ], dtype=complex)
    closed_RZ = np.array([
        [np.exp(-0.5j * theta), 0.0],
        [0.0, np.exp(0.5j * theta)]
    ], dtype=complex)
    assert np.allclose(closed_RY.conj().T @ closed_RY, I_mat, atol=1e-15), "Failed RY unitarity"
    assert np.allclose(closed_RZ.conj().T @ closed_RZ, I_mat, atol=1e-15), "Failed RZ unitarity"

print("[VERIFIED 3/5] Unitarity U^dag * U = I holds within machine precision (atol=1e-15).")

# 4. Mixed-Product Property: (A (x) B)(C (x) D) == (AC) (x) (BD)
A = X_mat
B = Z_mat
C = Y_mat
D = X_mat
lhs = np.kron(A, B) @ np.kron(C, D)
rhs = np.kron(A @ C, B @ D)
assert np.allclose(lhs, rhs, atol=1e-15), "Failed Mixed-Product property assertion!"
print("[VERIFIED 4/5] Multi-qubit Kronecker mixed-product property (A(x)B)(C(x)D) == (AC)(x)(BD) verified.")

# 5. Parameter-Shift Trigonometric Derivative Identity
# For f(theta) = A + B*cos(theta) + C*sin(theta)
np.random.seed(SEED)
A_val = 0.42
B_val = -0.73
C_val = 0.58

def test_f(t):
    return A_val + B_val * np.cos(t) + C_val * np.sin(t)

def test_f_prime(t):
    return - B_val * np.sin(t) + C_val * np.cos(t)

for theta in test_angles:
    exact_grad = test_f_prime(theta)
    param_shift_grad = 0.5 * (test_f(theta + np.pi / 2) - test_f(theta - np.pi / 2))
    abs_diff = np.abs(exact_grad - param_shift_grad)
    assert abs_diff < 1e-15, f"Parameter shift mismatch at theta={theta}: diff={abs_diff}"

print("[VERIFIED 5/5] Parameter-shift rule produces exact analytical derivatives (error < 1e-15).")
print("\n[ALL MATHEMATICAL FOUNDATIONS VERIFIED SUCCESSFULLY]")


[VERIFIED 1/5] Pauli matrix algebraic and commutation identities hold exactly.
[VERIFIED 2/5] Matrix exponentials exp(-i*theta/2*P) match closed-form RY and RZ gates.
[VERIFIED 3/5] Unitarity U^dag * U = I holds within machine precision (atol=1e-15).
[VERIFIED 4/5] Multi-qubit Kronecker mixed-product property (A(x)B)(C(x)D) == (AC)(x)(BD) verified.
[VERIFIED 5/5] Parameter-shift rule produces exact analytical derivatives (error < 1e-15).

[ALL MATHEMATICAL FOUNDATIONS VERIFIED SUCCESSFULLY]


### Interpretation of Mathematical Foundations Verification
The numerical verification confirms the absolute validity of our foundational mathematical structures:
1. The Pauli operator algebra satisfies all canonical commutation and anti-commutation identities.
2. The closed-form trigonometric representations of $R_Y(\theta)$ and $R_Z(\theta)$ are exact representations of the Lie-algebraic matrix exponentials $\exp(-i \frac{\theta}{2} P)$, exhibiting strict unitarity ($U^\dagger U = I$) to within machine precision ($< 10^{-15}$).
3. Multi-qubit operator composition via `np.kron` satisfies the mixed-product theorem, guaranteeing that gate application on individual tensor factors correctly models parallel quantum operations.
4. The parameter-shift rule was verified to be mathematically exact for any function generated by Pauli rotations, confirming that macroscopic shifts ($s = \pi/2$) evaluate analytical gradients without the truncation errors or numerical cancellation inherent in finite-difference approximations.

With the research objectives, configuration, and mathematical foundations established and validated, the experiment is ready for Section 04.


# 04 — Quantum State Representation

## 4.1 Concepts & Mathematical Foundations
In this section, we construct the state-vector representation from scratch.

### Complex Probability Amplitudes
A quantum state vector $|\psi\rangle$ is a complex unit vector in $\mathbb{C}^{2^n}$. Unlike classical probability distributions, where components $p_i \in [0, 1]$ represent mutually exclusive non-negative probabilities summing to 1, quantum states assign a **complex amplitude** $c_i \in \mathbb{C}$ to each basis configuration:
$$c_i = a_i + i b_i = r_i e^{i \phi_i}$$

Complex amplitudes are the fundamental mechanism enabling:
1. **Quantum Superposition:** A quantum system simultaneously occupies a linear combination of all computational basis configurations.
2. **Quantum Interference:** When states evolve through unitary gates, amplitudes can add constructively ($e^{i \phi_1} + e^{i \phi_2} > 0$) or destructively ($e^{i \phi} + e^{i (\phi + \pi)} = 0$), allowing quantum algorithms to cancel unpromising outcomes and amplify target solutions.

### State Dimension & Exponential Hilbert Space
For an $n$-qubit register, the number of distinct binary configurations is $2^n$. The state vector must account for every possible joint binary string:
$$\dim(\mathcal{H}) = 2^n$$
For our 2-qubit system ($n=2$):
$$\dim(\mathcal{H}) = 2^2 = 4$$

### Explicit Basis Ordering Convention
We establish a fixed, one-dimensional complex vector convention of shape `(4,)` with `dtype=np.complex128`:
$$|00\rangle = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix}, \quad
|01\rangle = \begin{pmatrix} 0 \\ 1 \\ 0 \\ 0 \end{pmatrix}, \quad
|10\rangle = \begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix}, \quad
|11\rangle = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$

Index $i \in \{0, 1, 2, 3\}$ corresponds to the decimal value of binary string $q_0 q_1$:
$$i = 2 q_0 + q_1$$
This 1D array representation is maintained uniformly throughout the entire notebook.

### Normalization Condition (Born's Rule)
Under Born's rule, the probability of obtaining outcome $i$ upon projective measurement in the computational basis is $P(i) = |c_i|^2 = c_i^* c_i$. Conservation of probability demands:
$$\langle \psi | \psi \rangle = \sum_{i=0}^{2^n - 1} |c_i|^2 = 1.0$$


In [4]:
# ---------------------------------------------------------------------------
# Section 04 Implementation: Quantum State Vectors & Utilities
# ---------------------------------------------------------------------------

def get_basis_state(label: str) -> np.ndarray:
    """
    Constructs a 1D complex basis state vector for a specified computational basis label.
    
    Parameters
    ----------
    label : str
        Binary string representing computational basis state ('0', '1', '00', '01', '10', '11').
        
    Returns
    -------
    np.ndarray
        1D complex128 NumPy array representing normalized computational basis vector.
    """
    n = len(label)
    dim = 2 ** n
    idx = int(label, 2)
    state = np.zeros(dim, dtype=np.complex128)
    state[idx] = 1.0 + 0.0j
    return state


def get_zero_state(n_qubits: int = 2) -> np.ndarray:
    """
    Constructs the initial computational ground state |0...0> for an n-qubit system.
    
    Parameters
    ----------
    n_qubits : int, default=2
        Number of qubits in the quantum register.
        
    Returns
    -------
    np.ndarray
        1D complex128 array of length 2^n with 1.0 at index 0 and 0.0 elsewhere.
    """
    dim = 2 ** n_qubits
    state = np.zeros(dim, dtype=np.complex128)
    state[0] = 1.0 + 0.0j
    return state


def is_normalized(state: np.ndarray, tol: float = 1e-12) -> bool:
    """
    Verifies whether a pure quantum state vector is normalized to unity (<psi|psi> == 1).
    
    Parameters
    ----------
    state : np.ndarray
        1D complex quantum state vector.
    tol : float, default=1e-12
        Numerical tolerance for floating-point comparison.
        
    Returns
    -------
    bool
        True if |<psi|psi> - 1| <= tol, False otherwise.
    """
    inner_product = np.vdot(state, state)
    return bool(np.abs(inner_product - 1.0) < tol)


def state_probabilities(state: np.ndarray) -> np.ndarray:
    """
    Calculates the computational basis probability distribution under Born's rule: P(i) = |c_i|^2.
    
    Parameters
    ----------
    state : np.ndarray
        1D complex quantum state vector.
        
    Returns
    -------
    np.ndarray
        1D real float64 array of measurement probabilities summing to 1.0.
    """
    return np.abs(state) ** 2


### Verification: Quantum State Representation

**What should be true:**
1. `get_zero_state(2)` must produce a 1D vector of shape `(4,)`, dtype `complex128`, matching $[1, 0, 0, 0]^T$.
2. All computational basis states (`|00>`, `|01>`, `|10>`, `|11>`) must be normalized: $\sum |c_i|^2 = 1.0$.
3. Computational basis states must be orthonormal: $\langle i | j \rangle = \delta_{ij}$.
4. Measuring `|00>` must yield $P(00) = 1.0$ and $P(01) = P(10) = P(11) = 0.0$.


In [5]:
# ---------------------------------------------------------------------------
# Section 04 Verification
# ---------------------------------------------------------------------------
psi_00 = get_zero_state(n_qubits=2)

# 1. Verify shape, dtype, and vector content
assert psi_00.shape == (4,), f"Failed: Expected shape (4,), got {psi_00.shape}"
assert psi_00.dtype == np.complex128, f"Failed: Expected complex128, got {psi_00.dtype}"
expected_00 = np.array([1.0 + 0.0j, 0.0 + 0.0j, 0.0 + 0.0j, 0.0 + 0.0j], dtype=np.complex128)
assert np.allclose(psi_00, expected_00), "Failed: |00> state vector does not match expected amplitudes."

# 2. Verify normalization
assert is_normalized(psi_00), "Failed: Ground state |00> is not normalized."

# 3. Verify all 4 basis states and orthonormality
basis_labels = ['00', '01', '10', '11']
states = [get_basis_state(lbl) for lbl in basis_labels]

for idx, (lbl, s) in enumerate(zip(basis_labels, states)):
    assert s.shape == (4,), f"Failed shape for |{lbl}>"
    assert is_normalized(s), f"Failed normalization for |{lbl}>"
    probs = state_probabilities(s)
    assert np.isclose(probs[idx], 1.0), f"Failed probability for |{lbl}>: P({idx}) != 1.0"
    assert np.isclose(np.sum(probs), 1.0), f"Failed total probability sum for |{lbl}>"

# Check mutual orthogonality <i|j> = delta_ij
for i in range(4):
    for j in range(4):
        inner = np.vdot(states[i], states[j])
        expected_inner = 1.0 if i == j else 0.0
        assert np.isclose(inner, expected_inner, atol=1e-15), f"Failed orthogonality between {basis_labels[i]} and {basis_labels[j]}"

print("[VERIFIED 1/3] State dimensions, complex dtype, and vector contents match ground state |00>.")
print("[VERIFIED 2/3] All computational basis states are strictly normalized and mutually orthonormal.")
print(f"[VERIFIED 3/3] Born's rule probability for |00>: P = {state_probabilities(psi_00)}")
print("\n[SECTION 04 VERIFICATION PASSED]")


[VERIFIED 1/3] State dimensions, complex dtype, and vector contents match ground state |00>.
[VERIFIED 2/3] All computational basis states are strictly normalized and mutually orthonormal.
[VERIFIED 3/3] Born's rule probability for |00>: P = [1. 0. 0. 0.]

[SECTION 04 VERIFICATION PASSED]


### Interpretation of State Representation
The verification demonstrates that our 1D complex state vector implementation correctly models the $2^2 = 4$ dimensional Hilbert space. Ground state $|00\rangle$ and all basis states satisfy the normalization condition $\sum |c_i|^2 = 1.0$ and are mutually orthonormal within machine precision ($10^{-15}$).


# 05 — Quantum Gates

## 5.1 Single-Qubit Unitary Operators
Every single-qubit quantum gate is represented by a $2 \times 2$ complex matrix $U \in \mathbb{C}^{2 \times 2}$ satisfying the **unitarity condition**:
$$U^\dagger U = U U^\dagger = I_2$$
where $U^\dagger = (U^*)^T$ is the conjugate transpose (Hermitian adjoint). Unitarity guarantees that gate transformations preserve quantum state norm: $\langle \psi | U^\dagger U | \psi \rangle = \langle \psi | \psi \rangle = 1$.

### Mathematical Definitions of Fundamental Gates:
1. **Identity Gate ($I$):**
   $$I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}$$
   Leaves any state invariant: $I |\psi\rangle = |\psi\rangle$.

2. **Pauli-$X$ Gate (Bit-Flip):**
   $$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$
   Flips computational basis states: $X |0\rangle = |1\rangle, \quad X |1\rangle = |0\rangle$.

3. **Pauli-$Y$ Gate (Bit-and-Phase Flip):**
   $$Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$$
   Applies simultaneous bit and phase flip: $Y |0\rangle = i |1\rangle, \quad Y |1\rangle = -i |0\rangle$.

4. **Pauli-$Z$ Gate (Phase-Flip):**
   $$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$
   Applies relative phase shift of $\pi$: $Z |0\rangle = |0\rangle, \quad Z |1\rangle = -|1\rangle$.

5. **Hadamard Gate ($H$):**
   $$H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$
   Creates equal superpositions: $H |0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle, \quad H |1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle$.

6. **Parameterized $R_Y(\theta)$ Rotation Gate:**
   $$R_Y(\theta) = \exp\left(-i \frac{\theta}{2} Y\right) = \begin{pmatrix} \cos(\theta/2) & -\sin(\theta/2) \\ \sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$$
   Rotates the state vector around the $Y$-axis of the Bloch sphere by angle $\theta$. Crucially, $R_Y(\theta)$ is strictly real-valued.

7. **Parameterized $R_Z(\theta)$ Rotation Gate:**
   $$R_Z(\theta) = \exp\left(-i \frac{\theta}{2} Z\right) = \begin{pmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2} \end{pmatrix}$$
   Rotates the state vector around the $Z$-axis of the Bloch sphere, introducing a continuous relative phase shift.


In [6]:
# ---------------------------------------------------------------------------
# Section 05 Implementation: Single-Qubit Quantum Gates
# ---------------------------------------------------------------------------

def check_unitary(U: np.ndarray, tol: float = 1e-14) -> bool:
    """
    Verifies whether a matrix U is unitary: U^dag @ U == I within numerical tolerance.
    
    Parameters
    ----------
    U : np.ndarray
        2D complex matrix to test.
    tol : float, default=1e-14
        Numerical tolerance for floating-point comparison.
        
    Returns
    -------
    bool
        True if unitary, False otherwise.
    """
    dim = U.shape[0]
    I_ref = np.eye(dim, dtype=np.complex128)
    product = U.conj().T @ U
    return bool(np.allclose(product, I_ref, atol=tol))


def gate_I() -> np.ndarray:
    """Single-qubit Identity gate."""
    return np.array([[1.0, 0.0],
                     [0.0, 1.0]], dtype=np.complex128)


def gate_X() -> np.ndarray:
    """Single-qubit Pauli-X (Bit-flip) gate."""
    return np.array([[0.0, 1.0],
                     [1.0, 0.0]], dtype=np.complex128)


def gate_Y() -> np.ndarray:
    """Single-qubit Pauli-Y gate."""
    return np.array([[0.0, -1.0j],
                     [1.0j, 0.0]], dtype=np.complex128)


def gate_Z() -> np.ndarray:
    """Single-qubit Pauli-Z (Phase-flip) gate."""
    return np.array([[1.0,  0.0],
                     [0.0, -1.0]], dtype=np.complex128)


def gate_H() -> np.ndarray:
    """Single-qubit Hadamard gate."""
    inv_sqrt2 = 1.0 / np.sqrt(2.0)
    return np.array([[inv_sqrt2,  inv_sqrt2],
                     [inv_sqrt2, -inv_sqrt2]], dtype=np.complex128)


def gate_RY(theta: float) -> np.ndarray:
    """
    Parameterized single-qubit RY rotation gate: exp(-i * theta/2 * Y).
    
    Parameters
    ----------
    theta : float
        Rotation angle in radians.
        
    Returns
    -------
    np.ndarray
        2x2 complex128 matrix: [[cos(t/2), -sin(t/2)], [sin(t/2), cos(t/2)]].
    """
    half_t = theta / 2.0
    c = np.cos(half_t)
    s = np.sin(half_t)
    return np.array([[c, -s],
                     [s,  c]], dtype=np.complex128)


def gate_RZ(theta: float) -> np.ndarray:
    """
    Parameterized single-qubit RZ rotation gate: exp(-i * theta/2 * Z).
    
    Parameters
    ----------
    theta : float
        Rotation angle in radians.
        
    Returns
    -------
    np.ndarray
        2x2 complex128 matrix: diag(exp(-i*t/2), exp(i*t/2)).
    """
    half_t = theta / 2.0
    return np.array([[np.exp(-1.0j * half_t), 0.0],
                     [0.0, np.exp(1.0j * half_t)]], dtype=np.complex128)


### Verification: Quantum Gates

**What should be true:**
1. All gates must have shape `(2, 2)` and complex dtype.
2. All fixed gates ($I, X, Y, Z, H$) must be unitary: $U^\dagger U = I$.
3. Parameterized gates $R_Y(\theta)$ and $R_Z(\theta)$ must be strictly unitary for multiple test angles $\theta \in \{0, \pi/6, \pi/4, \pi/2, \pi, 2.5, 5.0\}$.
4. Known gate actions on computational basis states $|0\rangle = [1, 0]^T$ and $|1\rangle = [0, 1]^T$:
   - $X |0\rangle = |1\rangle$ and $X |1\rangle = |0\rangle$
   - $Z |0\rangle = |0\rangle$ and $Z |1\rangle = -|1\rangle$
   - $H |0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$
   - $R_Y(0) = I$ and $R_Z(0) = I$
   - $R_Y(\pi) |0\rangle = |1\rangle$ (up to numerical tolerance)


In [7]:
# ---------------------------------------------------------------------------
# Section 05 Verification
# ---------------------------------------------------------------------------
ket_0 = np.array([1.0, 0.0], dtype=np.complex128)
ket_1 = np.array([0.0, 1.0], dtype=np.complex128)

# 1. Unitarity of fixed gates
fixed_gates = {'I': gate_I(), 'X': gate_X(), 'Y': gate_Y(), 'Z': gate_Z(), 'H': gate_H()}
for name, G in fixed_gates.items():
    assert G.shape == (2, 2), f"Failed shape for gate {name}"
    assert check_unitary(G), f"Failed unitarity assertion for gate {name}"

print("[VERIFIED 1/4] Fixed gates (I, X, Y, Z, H) have dimension (2, 2) and are strictly unitary.")

# 2. Unitarity of parameterized rotation gates across varied angles
test_angles = [0.0, np.pi / 6, np.pi / 4, np.pi / 2, np.pi, 2.5, 5.0]
for t in test_angles:
    ry = gate_RY(t)
    rz = gate_RZ(t)
    assert ry.shape == (2, 2), f"Failed RY shape at theta={t}"
    assert rz.shape == (2, 2), f"Failed RZ shape at theta={t}"
    assert check_unitary(ry), f"Failed RY unitarity at theta={t}"
    assert check_unitary(rz), f"Failed RZ unitarity at theta={t}"

print("[VERIFIED 2/4] Parameterized rotation gates RY(theta) and RZ(theta) are unitary across test angles.")

# 3. Canonical gate transformations
assert np.allclose(gate_X() @ ket_0, ket_1), "Failed: X|0> != |1>"
assert np.allclose(gate_X() @ ket_1, ket_0), "Failed: X|1> != |0>"
assert np.allclose(gate_Z() @ ket_0, ket_0), "Failed: Z|0> != |0>"
assert np.allclose(gate_Z() @ ket_1, -ket_1), "Failed: Z|1> != -|1>"
assert np.allclose(gate_H() @ ket_0, (ket_0 + ket_1) / np.sqrt(2.0)), "Failed: H|0> != (|0>+|1>)/sqrt(2)"
print("[VERIFIED 3/4] Canonical state transformations X|0>=|1>, Z|0>=|0>, H|0>=(|0>+|1>)/sqrt(2) hold.")

# 4. Identity limit of rotations: RY(0) == I, RZ(0) == I, and full bit-flip RY(pi)|0> == |1>
assert np.allclose(gate_RY(0.0), gate_I()), "Failed: RY(0) != I"
assert np.allclose(gate_RZ(0.0), gate_I()), "Failed: RZ(0) != I"
assert np.allclose(gate_RY(np.pi) @ ket_0, ket_1, atol=1e-15), "Failed: RY(pi)|0> != |1>"
print("[VERIFIED 4/4] Rotation boundary conditions verified: RY(0)==I, RZ(0)==I, RY(pi)|0>==|1>.")
print("\n[SECTION 05 VERIFICATION PASSED]")


[VERIFIED 1/4] Fixed gates (I, X, Y, Z, H) have dimension (2, 2) and are strictly unitary.
[VERIFIED 2/4] Parameterized rotation gates RY(theta) and RZ(theta) are unitary across test angles.
[VERIFIED 3/4] Canonical state transformations X|0>=|1>, Z|0>=|0>, H|0>=(|0>+|1>)/sqrt(2) hold.
[VERIFIED 4/4] Rotation boundary conditions verified: RY(0)==I, RZ(0)==I, RY(pi)|0>==|1>.

[SECTION 05 VERIFICATION PASSED]


### Interpretation of Quantum Gate Implementation
All single-qubit gates satisfy strict numerical unitarity ($U^\dagger U = I$, machine precision $< 10^{-14}$). Parameterized gates smoothly deform states as rotation angles vary and recover the identity operator at $\theta = 0$, validating their mathematical correctness for variational optimization.


# 06 — Multi-Qubit Operations

## 6.1 Multi-Qubit Operators & The Kronecker Product
When single-qubit operations act in parallel on disjoint qubits, the joint multi-qubit operator is constructed via the Kronecker product:
$$A \otimes B$$
Using `np.kron(A, B)`, if $A$ acts on Qubit 0 and $B$ acts on Qubit 1, the resulting $4 \times 4$ operator transforms the 2-qubit state according to our established convention:
$$(A \otimes B) (|q_0\rangle \otimes |q_1\rangle) = (A |q_0\rangle) \otimes (B |q_1\rangle)$$

### Qubit Ordering Convention Reiteration:
- **First factor (left / most significant):** Qubit 0 ($q_0$)
- **Second factor (right / least significant):** Qubit 1 ($q_1$)
- **Basis Order:** $|00\rangle, |01\rangle, |10\rangle, |11\rangle$

### The Controlled-NOT (CNOT) Entangling Gate
A 2-qubit gate cannot always be decomposed into a tensor product of single-qubit gates. The canonical entangling gate is the Controlled-NOT (CNOT) gate.

With **Qubit 0 as Control** and **Qubit 1 as Target** ($CNOT_{0 \to 1}$):
- If Qubit 0 is in state $|0\rangle$, Qubit 1 is unchanged ($I$ applied).
- If Qubit 0 is in state $|1\rangle$, Qubit 1 is flipped ($X$ applied).

Algebraically:
$$CNOT_{0 \to 1} = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes X$$
In matrix form under basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$:
$$CNOT_{0 \to 1} = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix} \otimes \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} + \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix} \otimes \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0
\end{pmatrix}$$

Action on computational basis states:
$$CNOT_{0 \to 1} |00\rangle = |00\rangle$$
$$CNOT_{0 \to 1} |01\rangle = |01\rangle$$
$$CNOT_{0 \to 1} |10\rangle = |11\rangle$$
$$CNOT_{0 \to 1} |11\rangle = |10\rangle$$


In [8]:
# ---------------------------------------------------------------------------
# Section 06 Implementation: Multi-Qubit Operations
# ---------------------------------------------------------------------------

def tensor_product(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Computes the Kronecker / tensor product A (x) B of two operators.
    
    Parameters
    ----------
    A : np.ndarray
        Operator acting on Qubit 0.
    B : np.ndarray
        Operator acting on Qubit 1.
        
    Returns
    -------
    np.ndarray
        Joint operator of dimension (dim(A)*dim(B), dim(A)*dim(B)).
    """
    return np.kron(A, B)


def gate_CNOT() -> np.ndarray:
    """
    Constructs the 2-qubit Controlled-NOT gate with Qubit 0 as Control and Qubit 1 as Target.
    
    Returns
    -------
    np.ndarray
        4x4 complex128 unitary matrix: |0><0| (x) I + |1><1| (x) X.
    """
    proj_0 = np.array([[1.0, 0.0],
                       [0.0, 0.0]], dtype=np.complex128)  # |0><0|
    proj_1 = np.array([[0.0, 0.0],
                       [0.0, 1.0]], dtype=np.complex128)  # |1><1|
    
    term_0 = tensor_product(proj_0, gate_I())
    term_1 = tensor_product(proj_1, gate_X())
    return term_0 + term_1


# 2-Qubit Observables constructed via tensor product
OBS_Z0 = tensor_product(gate_Z(), gate_I())  # Pauli-Z on Qubit 0
OBS_Z1 = tensor_product(gate_I(), gate_Z())  # Pauli-Z on Qubit 1


### Verification: Multi-Qubit Operations

**What should be true:**
1. The tensor product of two $2 \times 2$ matrices must yield a $4 \times 4$ matrix.
2. Multi-qubit Pauli observables ($I \otimes I, X \otimes I, I \otimes X, Z \otimes I, I \otimes Z$) must have shape `(4, 4)` and be unitary and Hermitian.
3. `gate_CNOT()` must be $4 \times 4$ and unitary ($CNOT^\dagger CNOT = I_4$).
4. CNOT must transform computational basis states exactly as defined:
   - $|00\rangle \to |00\rangle$
   - $|01\rangle \to |01\rangle$
   - $|10\rangle \to |11\rangle$
   - $|11\rangle \to |10\rangle$
5. When CNOT acts on an entangled or superposition state, the resulting state must remain strictly normalized.


In [9]:
# ---------------------------------------------------------------------------
# Section 06 Verification
# ---------------------------------------------------------------------------
cnot = gate_CNOT()

# 1. Dimension and unitarity of CNOT
assert cnot.shape == (4, 4), f"Failed: CNOT shape must be (4, 4), got {cnot.shape}"
assert check_unitary(cnot), "Failed: CNOT is not unitary."

# 2. Verify CNOT on all computational basis states
basis_00 = get_basis_state('00')
basis_01 = get_basis_state('01')
basis_10 = get_basis_state('10')
basis_11 = get_basis_state('11')

assert np.allclose(cnot @ basis_00, basis_00), "Failed: CNOT|00> != |00>"
assert np.allclose(cnot @ basis_01, basis_01), "Failed: CNOT|01> != |01>"
assert np.allclose(cnot @ basis_10, basis_11), "Failed: CNOT|10> != |11>"
assert np.allclose(cnot @ basis_11, basis_10), "Failed: CNOT|11> != |10>"
print("[VERIFIED 1/4] CNOT truth table on computational basis states matches theoretical specification.")

# 3. Verify multi-qubit Pauli tensor products
multi_paulis = {
    'I(x)I': tensor_product(gate_I(), gate_I()),
    'X(x)I': tensor_product(gate_X(), gate_I()),
    'I(x)X': tensor_product(gate_I(), gate_X()),
    'Z(x)I': OBS_Z0,
    'I(x)Z': OBS_Z1
}

for name, op in multi_paulis.items():
    assert op.shape == (4, 4), f"Failed shape for {name}"
    assert check_unitary(op), f"Failed unitarity for {name}"
    # Hermitian check: op^dag == op
    assert np.allclose(op.conj().T, op), f"Failed Hermiticity for {name}"

print("[VERIFIED 2/4] Multi-qubit Pauli operators (I(x)I, X(x)I, I(x)X, Z(x)I, I(x)Z) are unitary & Hermitian.")

# 4. Entanglement creation: H on q0 followed by CNOT produces Bell state |Phi+> = (|00> + |11>)/sqrt(2)
h_q0 = tensor_product(gate_H(), gate_I())
superpos = h_q0 @ basis_00
bell_state = cnot @ superpos
expected_bell = (basis_00 + basis_11) / np.sqrt(2.0)
assert np.allclose(bell_state, expected_bell), "Failed: Bell state creation failed."
assert is_normalized(bell_state), "Failed: Bell state is not normalized."
print("[VERIFIED 3/4] Entanglement generation verified: CNOT(H (x) I)|00> yields normalized Bell state |Phi+>.")

# 5. Diagonal structure of Z0 and Z1
assert np.allclose(np.diag(OBS_Z0), [1.0, 1.0, -1.0, -1.0]), "Failed: Z0 diagonal mismatch."
assert np.allclose(np.diag(OBS_Z1), [1.0, -1.0, 1.0, -1.0]), "Failed: Z1 diagonal mismatch."
print("[VERIFIED 4/4] Observable matrix diagonals match qubit ordering conventions.")
print("\n[SECTION 06 VERIFICATION PASSED]")


[VERIFIED 1/4] CNOT truth table on computational basis states matches theoretical specification.
[VERIFIED 2/4] Multi-qubit Pauli operators (I(x)I, X(x)I, I(x)X, Z(x)I, I(x)Z) are unitary & Hermitian.
[VERIFIED 3/4] Entanglement generation verified: CNOT(H (x) I)|00> yields normalized Bell state |Phi+>.
[VERIFIED 4/4] Observable matrix diagonals match qubit ordering conventions.

[SECTION 06 VERIFICATION PASSED]


### Interpretation of Multi-Qubit Operations
The Kronecker product implementation and CNOT matrix conform strictly to the specified qubit ordering ($q_0$ control, $q_1$ target). Entanglement generation was confirmed by synthesizing the maximally entangled Bell state $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$, with strict preservation of norm and unitarity.


# 07 — Feature Map

## 7.1 Classical-to-Quantum Feature Encoding
To process continuous classical data with a quantum circuit, classical features must be mapped into quantum state amplitudes. In our baseline design, we utilize **$R_Y$ product angle encoding**:
$$U_{\text{enc}}(x) = R_Y(x_1) \otimes R_Y(x_2)$$
where $x = (x_1, x_2)^T \in \mathbb{R}^2$.

### Encoding Dynamics:
Starting from the ground state $|00\rangle = |0\rangle \otimes |0\rangle$:
$$|\psi(x)\rangle = U_{\text{enc}}(x) |00\rangle = (R_Y(x_1) |0\rangle) \otimes (R_Y(x_2) |0\rangle)$$

For a single qubit, rotation by angle $x_k$ acts on $|0\rangle$ as:
$$R_Y(x_k) |0\rangle = \cos\left(\frac{x_k}{2}\right) |0\rangle + \sin\left(\frac{x_k}{2}\right) |1\rangle$$

Expanding the 2-qubit tensor product:
$$|\psi(x)\rangle = \begin{pmatrix}
\cos(x_1 / 2) \cos(x_2 / 2) \\
\cos(x_1 / 2) \sin(x_2 / 2) \\
\sin(x_1 / 2) \cos(x_2 / 2) \\
\sin(x_1 / 2) \sin(x_2 / 2)
\end{pmatrix}$$

### Scientific Justifications & Design Principles:
1. **One Feature per Qubit:** Direct one-to-one mapping ($x_1 \to q_0, x_2 \to q_1$) prevents multi-feature cross-talk during encoding, ensuring that non-linear interaction between features is introduced strictly by the downstream variational ansatz.
2. **Data Encoding Parameters vs. Trainable Parameters:** The encoding parameters $x$ are fixed classical data samples and are **never** updated during optimization. Gradients are taken solely with respect to the variational ansatz parameters $\theta$.
3. **Periodicity & Scaling:** Because $R_Y(x_k)$ is $4\pi$-periodic (or $2\pi$-periodic up to global phase), unscaled features with large ranges could wrap around the Bloch sphere, inducing non-monotonic aliasing. When datasets are ingested in Section 15, features will be scaled into an appropriate interval (such as $[-\pi/2, \pi/2]$ or $[0, \pi]$). At this stage, the feature map accepts raw float values.


In [10]:
# ---------------------------------------------------------------------------
# Section 07 Implementation: Feature Map
# ---------------------------------------------------------------------------

def feature_map(x: np.ndarray) -> np.ndarray:
    """
    Encodes a 2D continuous classical feature vector x = [x1, x2] into a 2-qubit state |psi(x)>
    using product RY angle encoding: U_enc(x) = RY(x1) (x) RY(x2).
    
    Parameters
    ----------
    x : np.ndarray or sequence
        Classical input vector containing exactly 2 continuous features.
        
    Returns
    -------
    np.ndarray
        1D complex128 state vector of length 4 representing |psi(x)> = U_enc(x)|00>.
    """
    x_arr = np.asarray(x, dtype=float)
    if x_arr.ndim != 1 or x_arr.shape[0] != 2:
        raise ValueError(f"Feature map requires a 1D vector of exactly 2 features, but received shape {x_arr.shape}.")
    
    x1, x2 = x_arr[0], x_arr[1]
    
    # Construct single-qubit RY gates
    ry1 = gate_RY(x1)
    ry2 = gate_RY(x2)
    
    # Multi-qubit encoding unitary U_enc(x) = RY(x1) (x) RY(x2)
    u_enc = tensor_product(ry1, ry2)
    
    # Apply to initial ground state |00>
    psi_00 = get_zero_state(n_qubits=2)
    encoded_state = u_enc @ psi_00
    return encoded_state


### Verification: Feature Map

**What should be true:**
1. `feature_map(x)` must return a vector of shape `(4,)`, dtype `complex128`, and be normalized.
2. Boundary input $[0, 0]$ must leave the system in $|00\rangle$: `[1, 0, 0, 0]`.
3. Input $[\pi, 0]$ must rotate Qubit 0 to $|1\rangle$, producing $|10\rangle$: `[0, 0, 1, 0]`.
4. Input $[0, \pi]$ must rotate Qubit 1 to $|1\rangle$, producing $|01\rangle$: `[0, 1, 0, 0]`.
5. Input $[\pi, \pi]$ must rotate both qubits to $|1\rangle$, producing $|11\rangle$: `[0, 0, 0, 1]`.
6. Non-trivial inputs (e.g., $[\pi/3, \pi/4]$) must produce analytical amplitudes matching $\cos(x_1/2)\cos(x_2/2)$, etc.


In [11]:
# ---------------------------------------------------------------------------
# Section 07 Verification
# ---------------------------------------------------------------------------
# 1. Boundary condition tests
psi_0_0 = feature_map([0.0, 0.0])
assert is_normalized(psi_0_0), "Failed normalization for [0, 0]"
assert np.allclose(psi_0_0, get_basis_state('00')), "Failed: feature_map([0, 0]) != |00>"

psi_pi_0 = feature_map([np.pi, 0.0])
assert is_normalized(psi_pi_0), "Failed normalization for [pi, 0]"
assert np.allclose(psi_pi_0, get_basis_state('10')), "Failed: feature_map([pi, 0]) != |10>"

psi_0_pi = feature_map([0.0, np.pi])
assert is_normalized(psi_0_pi), "Failed normalization for [0, pi]"
assert np.allclose(psi_0_pi, get_basis_state('01')), "Failed: feature_map([0, pi]) != |01>"

psi_pi_pi = feature_map([np.pi, np.pi])
assert is_normalized(psi_pi_pi), "Failed normalization for [pi, pi]"
assert np.allclose(psi_pi_pi, get_basis_state('11')), "Failed: feature_map([pi, pi]) != |11>"
print("[VERIFIED 1/3] Boundary encodings [0,0]->|00>, [pi,0]->|10>, [0,pi]->|01>, [pi,pi]->|11> pass.")

# 2. Analytical formula verification on continuous input
x_test = np.array([np.pi / 3.0, np.pi / 4.0])
psi_test = feature_map(x_test)
expected_amplitudes = np.array([
    np.cos(x_test[0] / 2) * np.cos(x_test[1] / 2),
    np.cos(x_test[0] / 2) * np.sin(x_test[1] / 2),
    np.sin(x_test[0] / 2) * np.cos(x_test[1] / 2),
    np.sin(x_test[0] / 2) * np.sin(x_test[1] / 2)
], dtype=np.complex128)

assert psi_test.shape == (4,), "Failed shape for feature map output"
assert is_normalized(psi_test), "Failed normalization for continuous input"
assert np.allclose(psi_test, expected_amplitudes, atol=1e-15), "Failed analytical amplitude formula check"
print("[VERIFIED 2/3] Continuous test input produces exact analytical amplitudes (atol=1e-15).")

# 3. Input validation check (must raise ValueError for incorrect dimensions)
try:
    feature_map([1.0, 2.0, 3.0])
    raise AssertionError("Failed: feature_map accepted 3 features instead of 2.")
except ValueError:
    pass
print("[VERIFIED 3/3] Feature map correctly rejects inputs with invalid dimensions.")
print("\n[SECTION 07 VERIFICATION PASSED]")


[VERIFIED 1/3] Boundary encodings [0,0]->|00>, [pi,0]->|10>, [0,pi]->|01>, [pi,pi]->|11> pass.
[VERIFIED 2/3] Continuous test input produces exact analytical amplitudes (atol=1e-15).
[VERIFIED 3/3] Feature map correctly rejects inputs with invalid dimensions.

[SECTION 07 VERIFICATION PASSED]


### Interpretation of Feature Map
The feature map verifies that continuous classical features are faithfully embedded into the 4-dimensional quantum state space. The orthogonal boundary conditions confirm our tensor product qubit indexing ($x_1 \to q_0$ as first factor, $x_2 \to q_1$ as second factor), preserving state normalization across all inputs.


# 08 — Variational Ansatz

## 8.1 Trainable Circuit Architecture
The variational ansatz is the parameterized quantum circuit $U(\theta)$ that transforms the encoded state $|\psi(x)\rangle$ into the classification state $|\psi(x, \theta)\rangle = U(\theta) |\psi(x)\rangle$.

### Layer Structure:
In our baseline configuration ($L = 1$), each layer consists of:
1. **Single-Qubit Rotations:** On each qubit $q \in \{0, 1\}$, an $R_Y$ rotation followed by an $R_Z$ rotation:
   $$U_{q}(\theta_{q, Y}, \theta_{q, Z}) = R_Z(\theta_{q, Z}) R_Y(\theta_{q, Y})$$
   *Note on matrix multiplication order:* In quantum state evolution $|\psi'\rangle = U_2 U_1 |\psi\rangle$, operator $U_1$ acts first and $U_2$ acts second. Therefore, applying $R_Y$ first and $R_Z$ second corresponds to matrix product $R_Z(\theta_Z) R_Y(\theta_Y)$.
2. **Entangling Gate:** A $CNOT_{0 \to 1}$ gate coupling Qubit 0 (control) and Qubit 1 (target).

The full unitary operator for layer $l$ is:
$$U_{\text{layer}}(\theta_l) = CNOT_{0 \to 1} \cdot \left[ \left( R_Z(\theta_{l, 1}) R_Y(\theta_{l, 0}) \right) \otimes \left( R_Z(\theta_{l, 3}) R_Y(\theta_{l, 2}) \right) \right]$$

For $L$ total layers, the overall ansatz unitary is the sequential composition:
$$U(\theta) = U_{\text{layer}}(\theta_L) \cdots U_{\text{layer}}(\theta_1)$$

### Explicit Parameter Layout:
All trainable parameters are organized into a flat 1D NumPy array of length $4L$:
- For $L=1$ (Baseline: 4 parameters):
  - $\theta_0$: Qubit 0, $R_Y$ rotation
  - $\theta_1$: Qubit 0, $R_Z$ rotation
  - $\theta_2$: Qubit 1, $R_Y$ rotation
  - $\theta_3$: Qubit 1, $R_Z$ rotation

### Expressibility vs. Trainability Trade-offs:
- **Expressibility:** Combining non-commuting rotations ($R_Y, R_Z$) allows parameterized exploration of the full single-qubit Bloch sphere ($SU(2)$). Adding CNOT entangles the state space, generating correlations across the 2-qubit Hilbert space.
- **Trainability & Depth:** A shallow ansatz ($L=1$) contains 4 parameters, minimizing parameter redundancy and optimization difficulty. Deeper circuits ($L=2, 3$) increase expressive reach but introduce non-convexity, potential saddle points, and trainability challenges (such as barren plateau precursors).


In [12]:
# ---------------------------------------------------------------------------
# Section 08 Implementation: Variational Ansatz
# ---------------------------------------------------------------------------

def variational_layer(theta_layer: np.ndarray) -> np.ndarray:
    """
    Constructs the 4x4 unitary matrix for a single variational layer given 4 parameters.
    
    Parameters
    ----------
    theta_layer : np.ndarray or sequence
        1D array containing 4 parameters: [theta_q0_y, theta_q0_z, theta_q1_y, theta_q1_z].
        
    Returns
    -------
    np.ndarray
        4x4 complex128 unitary matrix: CNOT @ ( (RZ(t1) @ RY(t0)) (x) (RZ(t3) @ RY(t2)) ).
    """
    params = np.asarray(theta_layer, dtype=float)
    if params.shape != (4,):
        raise ValueError(f"Each variational layer requires exactly 4 parameters, got shape {params.shape}.")
    
    # Qubit 0: RY(theta_0) then RZ(theta_1) -> Matrix: RZ(theta_1) @ RY(theta_0)
    u_q0 = gate_RZ(params[1]) @ gate_RY(params[0])
    
    # Qubit 1: RY(theta_2) then RZ(theta_3) -> Matrix: RZ(theta_3) @ RY(theta_2)
    u_q1 = gate_RZ(params[3]) @ gate_RY(params[2])
    
    # Parallel rotation operator
    u_rot = tensor_product(u_q0, u_q1)
    
    # Entangling CNOT gate
    u_layer = gate_CNOT() @ u_rot
    return u_layer


def variational_ansatz(theta: np.ndarray, n_layers: int = 1) -> np.ndarray:
    """
    Constructs the complete 4x4 unitary matrix for an n-layer variational ansatz.
    
    Parameters
    ----------
    theta : np.ndarray or sequence
        Flat 1D parameter array of length 4 * n_layers.
    n_layers : int, default=1
        Number of variational layers.
        
    Returns
    -------
    np.ndarray
        4x4 complex128 unitary matrix representing U(theta).
    """
    params = np.asarray(theta, dtype=float)
    expected_len = 4 * n_layers
    if params.shape != (expected_len,):
        raise ValueError(f"Expected {expected_len} parameters for {n_layers} layer(s), but got {params.shape[0]}.")
    
    # Initialize with 4x4 identity operator
    u_total = np.eye(4, dtype=np.complex128)
    
    for l in range(n_layers):
        layer_params = params[4 * l : 4 * (l + 1)]
        u_l = variational_layer(layer_params)
        # Apply subsequent layers: U_total = U_l @ U_total
        u_total = u_l @ u_total
        
    return u_total


### Verification: Variational Ansatz

**What should be true:**
1. Baseline ansatz requires exactly 4 parameters ($L=1$).
2. The layer unitary $U_{\text{layer}}(\theta)$ must be of shape `(4, 4)` and strictly unitary ($U^\dagger U = I$).
3. Multi-layer configurations ($L=1, 2, 3$) must produce unitary matrices of shape `(4, 4)` with parameter lengths 4, 8, and 12, respectively.
4. When applied to any normalized state vector, the resulting state must remain normalized.


In [13]:
# ---------------------------------------------------------------------------
# Section 08 Verification
# ---------------------------------------------------------------------------
# 1. Single-layer baseline verification
np.random.seed(SEED)
theta_baseline = np.random.uniform(-np.pi, np.pi, size=4)

u_ansatz_1 = variational_ansatz(theta_baseline, n_layers=1)
assert u_ansatz_1.shape == (4, 4), f"Failed shape: expected (4, 4), got {u_ansatz_1.shape}"
assert check_unitary(u_ansatz_1), "Failed unitarity for 1-layer ansatz."
print("[VERIFIED 1/3] 1-layer baseline ansatz (4 parameters) is dimensionally correct and unitary.")

# 2. Multi-layer scalability and unitarity (L=2: 8 params, L=3: 12 params)
for l in [1, 2, 3]:
    theta_l = np.random.uniform(-np.pi, np.pi, size=4 * l)
    u_l = variational_ansatz(theta_l, n_layers=l)
    assert u_l.shape == (4, 4), f"Failed shape for {l} layers."
    assert check_unitary(u_l), f"Failed unitarity for {l} layers."

print("[VERIFIED 2/3] Multi-layer ansatzes (L=1, 2, 3) maintain strict unitarity and correct parameter counts.")

# 3. Norm preservation on arbitrary input state
test_state = feature_map([0.5, -1.2])
evolved_state = u_ansatz_1 @ test_state
assert is_normalized(evolved_state), "Failed norm preservation after applying ansatz."
print("[VERIFIED 3/3] State normalization is strictly preserved under variational ansatz evolution.")
print("\n[SECTION 08 VERIFICATION PASSED]")


[VERIFIED 1/3] 1-layer baseline ansatz (4 parameters) is dimensionally correct and unitary.
[VERIFIED 2/3] Multi-layer ansatzes (L=1, 2, 3) maintain strict unitarity and correct parameter counts.
[VERIFIED 3/3] State normalization is strictly preserved under variational ansatz evolution.

[SECTION 08 VERIFICATION PASSED]


### Interpretation of Variational Ansatz
The variational ansatz implementation confirms that $U(\theta)$ acts as a valid, unitary multi-qubit operator. The parameter indexing is unambiguous, and the modular construction readily scales from $L=1$ to $L=3$ layers while preserving full unitarity.


# 09 — VQC Forward Pass

## 9.1 State Evolution Pipeline
The VQC forward pass connects classical feature encoding and variational state evolution into a single deterministic transformation:
$$x \xrightarrow{U_{\text{enc}}(x)} |\psi(x)\rangle \xrightarrow{U(\theta)} |\psi(x, \theta)\rangle$$

Algebraically:
$$|\psi(x, \theta)\rangle = U(\theta) U_{\text{enc}}(x) |00\rangle$$

### Modular Separation of Concerns:
In rigorous experimental quantum computing, the forward state preparation must remain completely decoupled from:
- Observable measurement,
- Loss evaluation,
- Gradient calculation,
- Optimizer updates.

The function `vqc_forward(x, theta)` computes purely quantum mechanical state evolution and returns the exact 4-dimensional complex state vector $|\psi(x, \theta)\rangle$.


In [14]:
# ---------------------------------------------------------------------------
# Section 09 Implementation: VQC Forward Pass
# ---------------------------------------------------------------------------

def vqc_forward(x: np.ndarray, theta: np.ndarray, n_layers: int = 1) -> np.ndarray:
    """
    Executes the forward quantum state evolution:
    |psi(x, theta)> = U(theta) @ U_enc(x) @ |00>.
    
    Parameters
    ----------
    x : np.ndarray
        2D classical input feature vector.
    theta : np.ndarray
        Trainable parameter vector of length 4 * n_layers.
    n_layers : int, default=1
        Number of variational layers.
        
    Returns
    -------
    np.ndarray
        1D complex128 state vector of length 4 representing |psi(x, theta)>.
    """
    # 1. Classical-to-quantum feature encoding: |psi(x)> = U_enc(x)|00>
    psi_encoded = feature_map(x)
    
    # 2. Parameterized variational circuit unitary: U(theta)
    u_variational = variational_ansatz(theta, n_layers=n_layers)
    
    # 3. Final state evolution: |psi(x, theta)> = U(theta)|psi(x)>
    final_state = u_variational @ psi_encoded
    return final_state


### Verification: VQC Forward Pass

**What should be true:**
1. The output state must be a 1D complex array of shape `(4,)` and be normalized.
2. **Zero-Parameter Reduction Check:** If all trainable parameters $\theta$ are zero ($\theta = \mathbf{0}$):
   - $R_Y(0) = I$ and $R_Z(0) = I$, so the rotation layer reduces to $I \otimes I = I_4$.
   - The variational circuit simplifies purely to $U(\mathbf{0}) = CNOT_{0 \to 1}$.
   - Thus, $|\psi(x, \mathbf{0})\rangle = CNOT_{0 \to 1} |\psi(x)\rangle$.
3. **Sensitivity Checks:**
   - Perturbing $x$ while holding $\theta$ constant must alter the state vector.
   - Perturbing $\theta$ while holding $x$ constant must alter the state vector.


In [15]:
# ---------------------------------------------------------------------------
# Section 09 Verification
# ---------------------------------------------------------------------------
x_sample = np.array([0.8, -0.4])
theta_sample = np.array([0.1, 0.2, -0.3, 0.4])

# 1. Forward pass shape, dtype, and normalization
psi_out = vqc_forward(x_sample, theta_sample, n_layers=1)
assert psi_out.shape == (4,), f"Failed: Expected shape (4,), got {psi_out.shape}"
assert psi_out.dtype == np.complex128, f"Failed: Expected complex128, got {psi_out.dtype}"
assert is_normalized(psi_out), "Failed: Forward pass output state is not normalized."
print("[VERIFIED 1/3] Forward state has shape (4,), complex128 dtype, and is strictly normalized.")

# 2. Zero-parameter reduction check: U(0) must reduce exactly to CNOT
theta_zeros = np.zeros(4)
psi_zeros = vqc_forward(x_sample, theta_zeros, n_layers=1)
expected_zeros = gate_CNOT() @ feature_map(x_sample)
assert np.allclose(psi_zeros, expected_zeros, atol=1e-15), "Failed: Zero-parameter reduction mismatch."
print("[VERIFIED 2/3] Zero-parameter state matches theoretical expectation: U(0)|psi(x)> == CNOT|psi(x)>.")

# 3. Sensitivity verification: forward pass is sensitive to both x and theta
psi_diff_x = vqc_forward(x_sample + np.array([0.1, 0.0]), theta_sample, n_layers=1)
psi_diff_theta = vqc_forward(x_sample, theta_sample + np.array([0.1, 0.0, 0.0, 0.0]), n_layers=1)

assert not np.allclose(psi_out, psi_diff_x), "Failed: Output state invariant to changes in input features x."
assert not np.allclose(psi_out, psi_diff_theta), "Failed: Output state invariant to changes in parameters theta."
print("[VERIFIED 3/3] Forward pass exhibits active sensitivity to feature inputs x and parameter weights theta.")
print("\n[SECTION 09 VERIFICATION PASSED]")


[VERIFIED 1/3] Forward state has shape (4,), complex128 dtype, and is strictly normalized.
[VERIFIED 2/3] Zero-parameter state matches theoretical expectation: U(0)|psi(x)> == CNOT|psi(x)>.
[VERIFIED 3/3] Forward pass exhibits active sensitivity to feature inputs x and parameter weights theta.

[SECTION 09 VERIFICATION PASSED]


### Interpretation of VQC Forward Pass
The forward state evolution is fully validated. The circuit properly executes the sequential mapping $x \to U_{\text{enc}}(x)|00\rangle \to U(\theta)|\psi(x)\rangle$. The zero-parameter sanity check confirms that at $\theta = \mathbf{0}$ all rotations become identity matrices and the ansatz acts strictly as CNOT.


# 10 — Measurement

## 10.1 Expectation Values & Classification Readout
Measurement is the final quantum step where quantum amplitudes are collapsed into classical numerical observables.

### Expectation Value Formulation:
For a normalized pure state $|\psi\rangle \in \mathbb{C}^4$ and a Hermitian observable operator $O = O^\dagger \in \mathbb{C}^{4 \times 4}$, the expectation value is:
$$\langle O \rangle = \langle \psi | O | \psi \rangle = \psi^\dagger O \psi$$
Because $O$ is Hermitian, $\langle O \rangle \in \mathbb{R}$ is strictly real. In floating-point arithmetic, any negligible imaginary residual ($< 10^{-14}$) resulting from finite-precision roundoff is safely discarded after assertion.

### Pauli-Z Observables on 2 Qubits:
1. **Pauli-$Z$ on Qubit 0 ($Z_0 = Z \otimes I$):**
   $$Z_0 = \text{diag}(+1, +1, -1, -1)$$
   Measures the longitudinal spin projection of Qubit 0. Expectation:
   $$\langle Z_0 \rangle = (|c_{00}|^2 + |c_{01}|^2) - (|c_{10}|^2 + |c_{11}|^2) \in [-1, +1]$$

2. **Pauli-$Z$ on Qubit 1 ($Z_1 = I \otimes Z$):**
   $$Z_1 = \text{diag}(+1, -1, +1, -1)$$
   Measures the longitudinal spin projection of Qubit 1. Expectation:
   $$\langle Z_1 \rangle = (|c_{00}|^2 + |c_{10}|^2) - (|c_{01}|^2 + |c_{11}|^2) \in [-1, +1]$$

3. **Average Pauli-$Z$ Observable ($\bar{Z}$):**
   $$\bar{Z} = \frac{1}{2} (Z_0 + Z_1) = \frac{1}{2} (Z \otimes I + I \otimes Z)$$
   $$\langle \bar{Z} \rangle = \frac{\langle Z_0 \rangle + \langle Z_1 \rangle}{2} \in [-1, +1]$$

### Classification Probability Mapping:
In binary classification, the objective is to predict class probability $\hat{y} = P(y=1|x) \in [0, 1]$. Because Pauli-$Z$ expectation values lie in the bipolar range $[-1, +1]$, we apply the standard affine mapping:
$$\hat{y} = \frac{\langle O \rangle + 1}{2}$$

- $\langle O \rangle = +1 \implies \hat{y} = 1.0$ (definite Class 1 outcome)
- $\langle O \rangle = -1 \implies \hat{y} = 0.0$ (definite Class 0 outcome)
- $\langle O \rangle = 0.0 \implies \hat{y} = 0.5$ (maximal uncertainty / classification decision boundary)

This linear rescaling preserves gradient linearity $\frac{\partial \hat{y}}{\partial \langle O \rangle} = \frac{1}{2}$ and enables seamless evaluation using Binary Cross-Entropy loss in Section 11.


In [16]:
# ---------------------------------------------------------------------------
# Section 10 Implementation: Quantum Measurement & Classification Mapping
# ---------------------------------------------------------------------------

def expectation_value(state: np.ndarray, observable: np.ndarray, tol_imag: float = 1e-12) -> float:
    """
    Computes the expectation value <psi|O|psi> for pure state |psi> and Hermitian operator O.
    
    Parameters
    ----------
    state : np.ndarray
        1D complex state vector.
    observable : np.ndarray
        2D Hermitian operator matrix.
    tol_imag : float, default=1e-12
        Maximum allowable magnitude of numerical imaginary residue.
        
    Returns
    -------
    float
        Real expectation value in R.
    """
    val = np.vdot(state, observable @ state)
    if np.abs(np.imag(val)) > tol_imag:
        raise ValueError(f"Observable expectation value has significant imaginary component: {np.imag(val)}")
    return float(np.real(val))


def z0_expectation(state: np.ndarray) -> float:
    """Computes expectation value of Pauli-Z on Qubit 0: <Z0> = <psi|(Z (x) I)|psi>."""
    return expectation_value(state, OBS_Z0)


def z1_expectation(state: np.ndarray) -> float:
    """Computes expectation value of Pauli-Z on Qubit 1: <Z1> = <psi|(I (x) Z)|psi>."""
    return expectation_value(state, OBS_Z1)


def average_z_expectation(state: np.ndarray) -> float:
    """Computes average expectation value across both qubits: (<Z0> + <Z1>) / 2."""
    return 0.5 * (z0_expectation(state) + z1_expectation(state))


def prediction_from_expectation(expectation: float) -> float:
    """
    Maps expectation value in [-1, +1] to binary classification prediction probability in [0, 1].
    
    Parameters
    ----------
    expectation : float
        Expectation value in range [-1, +1].
        
    Returns
    -------
    float
        Predicted probability in range [0, 1].
    """
    p = 0.5 * (expectation + 1.0)
    # Numerical safeguard to guarantee strict confinement to [0, 1]
    return float(np.clip(p, 0.0, 1.0))


### Verification: Quantum Measurement

**What should be true:**
1. Exact expectation values on computational basis states under our qubit ordering:
   - $|00\rangle$: $\langle Z_0 \rangle = +1$, $\langle Z_1 \rangle = +1$, $\langle \bar{Z} \rangle = +1$
   - $|01\rangle$: $\langle Z_0 \rangle = +1$, $\langle Z_1 \rangle = -1$, $\langle \bar{Z} \rangle = 0$
   - $|10\rangle$: $\langle Z_0 \rangle = -1$, $\langle Z_1 \rangle = +1$, $\langle \bar{Z} \rangle = 0$
   - $|11\rangle$: $\langle Z_0 \rangle = -1$, $\langle Z_1 \rangle = -1$, $\langle \bar{Z} \rangle = -1$
2. Expectation values on arbitrary pure states must lie strictly within $[-1.0, +1.0]$.
3. Probability mapping must map $-1 \to 0.0$, $0 \to 0.5$, and $+1 \to 1.0$.


In [17]:
# ---------------------------------------------------------------------------
# Section 10 Verification
# ---------------------------------------------------------------------------
# 1. Exact expectation values on computational basis states
basis_states = {
    '00': (get_basis_state('00'), +1.0, +1.0),
    '01': (get_basis_state('01'), +1.0, -1.0),
    '10': (get_basis_state('10'), -1.0, +1.0),
    '11': (get_basis_state('11'), -1.0, -1.0),
}

for lbl, (st, exp_z0, exp_z1) in basis_states.items():
    z0 = z0_expectation(st)
    z1 = z1_expectation(st)
    z_avg = average_z_expectation(st)
    assert np.isclose(z0, exp_z0), f"Failed <Z0> on |{lbl}>: expected {exp_z0}, got {z0}"
    assert np.isclose(z1, exp_z1), f"Failed <Z1> on |{lbl}>: expected {exp_z1}, got {z1}"
    assert np.isclose(z_avg, 0.5 * (exp_z0 + exp_z1)), f"Failed <Z_avg> on |{lbl}>"

print("[VERIFIED 1/3] Observable expectation values on all computational basis states match analytical truth.")

# 2. Boundedness verification on random superposition states
np.random.seed(SEED)
for _ in range(20):
    raw_amps = np.random.normal(size=4) + 1j * np.random.normal(size=4)
    rand_state = raw_amps / np.linalg.norm(raw_amps)
    z0 = z0_expectation(rand_state)
    z1 = z1_expectation(rand_state)
    assert -1.0 - 1e-12 <= z0 <= 1.0 + 1e-12, f"Z0 expectation out of bounds: {z0}"
    assert -1.0 - 1e-12 <= z1 <= 1.0 + 1e-12, f"Z1 expectation out of bounds: {z1}"

print("[VERIFIED 2/3] Expectation values strictly confined within physical bounds [-1.0, +1.0].")

# 3. Probability mapping tests
assert np.isclose(prediction_from_expectation(-1.0), 0.0), "Failed: map(-1) != 0.0"
assert np.isclose(prediction_from_expectation(0.0), 0.5), "Failed: map(0) != 0.5"
assert np.isclose(prediction_from_expectation(1.0), 1.0), "Failed: map(+1) != 1.0"
print("[VERIFIED 3/3] Prediction mapping [-1, +1] -> [0, 1] satisfies all boundary conditions.")
print("\n[SECTION 10 VERIFICATION PASSED]")


[VERIFIED 1/3] Observable expectation values on all computational basis states match analytical truth.
[VERIFIED 2/3] Expectation values strictly confined within physical bounds [-1.0, +1.0].
[VERIFIED 3/3] Prediction mapping [-1, +1] -> [0, 1] satisfies all boundary conditions.

[SECTION 10 VERIFICATION PASSED]


### Interpretation of Measurement Layer
The measurement layer confirms that expectation values for single-qubit Pauli observables $Z_0$ and $Z_1$ precisely follow the theoretical eigenvalues $+1$ and $-1$ across basis configurations. The probability mapping $\hat{y} = (\langle Z \rangle + 1)/2$ scales expectation values into valid probabilities $\hat{y} \in [0, 1]$ ready for loss function evaluation.


# Final Integration Test for Sections 04–10

To conclude the implementation of Sections 04 through 10, we conduct an end-to-end integration test of the complete forward pipeline:
$$\text{Classical Input } x \longrightarrow \text{Feature Map } |\psi(x)\rangle \longrightarrow \text{Ansatz } |\psi(x, \theta)\rangle \longrightarrow \text{Measurement } \langle Z_0 \rangle \longrightarrow \text{Prediction } \hat{y}$$

This integration test verifies that all data structures, matrix dimensions, state normalizations, and measurement bounds operate seamlessly in unison before proceeding to loss functions and optimization in later sections.


In [18]:
# ---------------------------------------------------------------------------
# End-to-End Integration Test: Sections 04 to 10
# ---------------------------------------------------------------------------
# Test configuration
x_test_sample = np.array([0.75, -1.25], dtype=float)
theta_test_sample = np.array([0.1, 0.2, 0.3, 0.4], dtype=float)

# Execute forward pipeline
psi_final = vqc_forward(x_test_sample, theta_test_sample, n_layers=1)

# Measurements
z0_val = z0_expectation(psi_final)
z1_val = z1_expectation(psi_final)
z_avg_val = average_z_expectation(psi_final)
pred_prob = prediction_from_expectation(z0_val)

# Log complete telemetry
print("=" * 70)
print("END-TO-END VQC FORWARD PASS TELEMETRY")
print("=" * 70)
print(f"Classical Input (x)              : {x_test_sample}")
print(f"Variational Parameters (theta)   : {theta_test_sample}")
print("-" * 70)
print(f"Final State Vector |psi(x,theta)>: ")
for idx, (lbl, amp) in enumerate(zip(['00', '01', '10', '11'], psi_final)):
    print(f"  |{lbl}> : {amp.real:+.6f} {amp.imag:+.6f}j  (Prob: {np.abs(amp)**2:.6f})")
print("-" * 70)
print(f"State Vector Dimension           : {psi_final.shape[0]} (Expected: 4)")
print(f"State Vector Norm (<psi|psi>)    : {np.vdot(psi_final, psi_final).real:.15f}")
print(f"Pauli-Z0 Expectation <Z0>        : {z0_val:+.6f}")
print(f"Pauli-Z1 Expectation <Z1>        : {z1_val:+.6f}")
print(f"Average Pauli-Z Expectation <Z>  : {z_avg_val:+.6f}")
print(f"Mapped Predicted Probability (y) : {pred_prob:.6f}")
print("=" * 70)

# Formal Assertions
assert psi_final.shape == (4,), "Integration Test Failed: Final state dimension != 4"
assert is_normalized(psi_final), "Integration Test Failed: Final state not normalized"
assert -1.0 <= z0_val <= 1.0, f"Integration Test Failed: <Z0> out of bounds: {z0_val}"
assert -1.0 <= z1_val <= 1.0, f"Integration Test Failed: <Z1> out of bounds: {z1_val}"
assert -1.0 <= z_avg_val <= 1.0, f"Integration Test Failed: <Z_avg> out of bounds: {z_avg_val}"
assert 0.0 <= pred_prob <= 1.0, f"Integration Test Failed: Prediction probability out of bounds: {pred_prob}"
assert not (np.isnan(pred_prob) or np.isinf(pred_prob)), "Integration Test Failed: NaN or Inf encountered"

print("\n[FINAL INTEGRATION TEST PASSED] All Section 04-10 components verified successfully.")


END-TO-END VQC FORWARD PASS TELEMETRY
Classical Input (x)              : [ 0.75 -1.25]
Variational Parameters (theta)   : [0.1 0.2 0.3 0.4]
----------------------------------------------------------------------
Final State Vector |psi(x,theta)>: 
  |00> : +0.773995 -0.239425j  (Prob: 0.656392)
  |01> : -0.414572 -0.041596j  (Prob: 0.173600)
  |10> : -0.180148 -0.055726j  (Prob: 0.035559)
  |11> : +0.364842 -0.036606j  (Prob: 0.134450)
----------------------------------------------------------------------
State Vector Dimension           : 4 (Expected: 4)
State Vector Norm (<psi|psi>)    : 1.000000000000000
Pauli-Z0 Expectation <Z0>        : +0.659983
Pauli-Z1 Expectation <Z1>        : +0.383901
Average Pauli-Z Expectation <Z>  : +0.521942
Mapped Predicted Probability (y) : 0.829992

[FINAL INTEGRATION TEST PASSED] All Section 04-10 components verified successfully.


### Summary of Sections 04–10 Implementation
The foundational quantum mechanical simulation engine has been fully established from first principles:
- **Section 04:** Complex 1D state vectors of shape `(4,)`, basis ordering $|00\rangle, |01\rangle, |10\rangle, |11\rangle$, and normalization utilities.
- **Section 05:** Complete suite of single-qubit unitary gates ($I, X, Y, Z, H, R_Y, R_Z$) with machine-precision unitarity verification.
- **Section 06:** Multi-qubit Kronecker composition and $CNOT_{0 \to 1}$ entangling operator.
- **Section 07:** Product $R_Y$ angle encoding feature map $U_{\text{enc}}(x)$.
- **Section 08:** Variational ansatz $U(\theta)$ with 4 parameters per layer and support for $L \in \{1, 2, 3\}$.
- **Section 09:** Decoupled forward quantum state evolution function `vqc_forward(x, theta)`.
- **Section 10:** Hermitian Pauli-$Z$ observable measurement and affine probability mapping $\hat{y} = (\langle Z \rangle + 1)/2$.

**STOPPING POINT:** Implementation halts strictly at Section 10 per instructions. Sections 11 (Loss Functions) through 26 will be constructed in subsequent phases upon review.
